<div style="border-left: 8px solid #04B4E3; padding: 0.25rem 0 0.25rem 1rem;">
<h1 style="color: #133C5A; margin-bottom: 0.25rem;">Expansão da Rede Federal de Educação Profissional e Economia Municipal</h1>
<p style="color: #00598E; margin: 0;"><strong>Notebook acadêmico principal</strong> · 2007–2019 · Município-ano</p>
</div>

**Pergunta de pesquisa.** A entrada em operação de unidades associadas à Fase II da expansão da Rede Federal alterou a atividade econômica dos municípios?

Este notebook apresenta a evidência construída até aqui. O período é 2007–2019, a unidade analítica é município-ano e o outcome econômico principal futuro é o pessoal ocupado assalariado (CEMPRE 708). O objetivo causal segue em avaliação; este documento não estima efeito causal.

## 1. Motivação e desenho

Na Fase II, municípios receberam unidades em anos diferentes. Trata-se, portanto, de um tratamento escalonado em um painel longitudinal. Uma etapa futura poderá avaliar um desenho de Diferenças-em-Diferenças com tratamento escalonado, possivelmente com o estimador de Callaway–Sant'Anna, somente se os gates de identificação forem atendidos.

## 2. Fontes de dados

| Fonte | Papel |
|---|---|
| MEC/SETEC | Identificação institucional da Fase II |
| INEP/Censo Escolar | Timing e atividade das unidades |
| IBGE/DTB | Existência territorial municipal |
| IBGE/CEMPRE | Atividade econômica e outcomes |
| Cadastro nacional da Rede Federal | Exposição e elegibilidade dos controles |

In [1]:
from pathlib import Path
import sys

import plotly.graph_objects as go
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import audita_populacao_causal_cempre as d12
from visualizacao_ipt import CORES_IPT, aplicar_tema_ipt, estilizar_tabela_ipt, salvar_figura_ipt

FIGURES = ROOT / 'outputs' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
painel = d12.carrega_painel_integrado()
diagnostico = d12.auditar_fase_ii(painel)
resumo_coortes = d12.resumo_por_coorte(painel)
controles_coortes = d12.auditar_controles_por_coorte(painel)
print(f'Painel carregado offline: {len(painel):,} município-ano.')

Painel carregado offline: 72,378 município-ano.


## 3. Universo e população

A construção abaixo separa exposição observada, elegibilidade estrutural de controles e candidatos principais. Os candidatos são uma população diagnóstica: **129 não é uma amostra causal final**.

In [2]:
municipios = painel.groupby('codigo_municipio_ibge', as_index=False).first()
populacao = pd.DataFrame({
    'etapa': ['Universo municipal', 'Expostos em algum momento', 'Nunca expostos', 'Controles estruturalmente elegíveis', 'Fase II', 'Candidatos principais'],
    'n_municipios': [
        municipios['codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['ever_treated'] == True, 'codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['sem_exposicao_observada_2007_2019'], 'codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['fl_elegivel_controle_candidato'], 'codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['fase_ii'] == True, 'codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['candidato_amostra_principal'] == True, 'codigo_municipio_ibge'].nunique(),
    ],
})
display(estilizar_tabela_ipt(populacao))
cores_populacao = [CORES_IPT['AZUL_ESCURO'], CORES_IPT['AZUL_MEDIO'], CORES_IPT['AZUL_CLARO'], CORES_IPT['CIANO'], CORES_IPT['AZUL_MEDIO'], CORES_IPT['AZUL_PRINCIPAL']]
fig = go.Figure(go.Bar(
    y=populacao['etapa'][::-1], x=populacao['n_municipios'][::-1], orientation='h',
    marker_color=cores_populacao[::-1], text=populacao['n_municipios'][::-1], textposition='outside',
    hovertemplate='<b>%{y}</b><br>%{x:,.0f} municípios<extra></extra>',
))
aplicar_tema_ipt(fig, titulo='Universo, exposição e populações diagnósticas')
fig.update_layout(showlegend=False)
fig.update_yaxes(showgrid=False)
fig.add_annotation(text='As categorias não formam um funil único', xref='paper', yref='paper', x=0, y=1.12, showarrow=False, font={'color': CORES_IPT['AZUL_MEDIO']})
salvar_figura_ipt(fig, FIGURES / '01_funil_populacao.png')
fig.show()

,etapa,n_municipios
0,Universo municipal,5570
1,Expostos em algum momento,147
2,Nunca expostos,4970
3,Controles estruturalmente elegíveis,4964
4,Fase II,147
5,Candidatos principais,129


### Interpretação

O universo contém todos os municípios observados no painel analítico. Os municípios nunca expostos não se confundem com os controles estruturais: estes últimos já incorporam regras de exclusão reproduzíveis. A população de 129 candidatos principais ainda poderá ser reduzida por decisões de identificação causal que não foram tomadas.

## 4. Painel CEMPRE

A camada técnica longa contém 506.870 linhas (5.570 municípios × 13 anos × 7 variáveis). O painel analítico contém apenas município-ano existentes territorialmente; 32 município-ano pré-criação territorial foram removidos desta camada, mas preservados na camada técnica. O outcome principal é o CEMPRE 708, pessoal ocupado assalariado.

In [3]:
resumo_painel = pd.DataFrame({
    'medida': ['Município-ano analítico', 'Municípios', 'Anos', 'Variáveis CEMPRE'],
    'valor': [len(painel), painel['codigo_municipio_ibge'].nunique(), f"{painel['ano'].min()}–{painel['ano'].max()}", 7],
})
display(estilizar_tabela_ipt(resumo_painel))

,medida,valor
0,Município-ano analítico,72378
1,Municípios,5570
2,Anos,2007–2019
3,Variáveis CEMPRE,7


### Interpretação

O painel tem cobertura longitudinal de 2007 a 2019 e preserva a unidade município-ano. A disponibilidade territorial é uma condição anterior à análise do outcome; por isso, a camada analítica não trata municípios ainda inexistentes como observações econômicas ausentes.

## 5. Coortes de tratamento

A coorte é o ano candidato de entrada em operação para cada município principal. Ela organiza o diagnóstico temporal, mas não produz por si só uma conclusão causal.

In [4]:
coortes = (diagnostico.loc[diagnostico['candidato_amostra_principal'] == True]
           .groupby('ano_coorte_candidata', as_index=False)
           .size().rename(columns={'size': 'n_candidatos'}))
coortes['ano_coorte_candidata'] = coortes['ano_coorte_candidata'].astype(int)
display(estilizar_tabela_ipt(coortes))
fig = go.Figure(go.Bar(
    x=coortes['ano_coorte_candidata'].astype(str), y=coortes['n_candidatos'],
    marker_color=CORES_IPT['AZUL_PRINCIPAL'], text=coortes['n_candidatos'], textposition='outside',
    hovertemplate='Coorte %{x}<br>%{y} candidatos principais<extra></extra>',
))
aplicar_tema_ipt(fig, titulo='Candidatos principais por coorte de tratamento')
fig.update_xaxes(title='Ano da coorte', type='category')
fig.update_yaxes(title='Número de municípios', rangemode='tozero')
salvar_figura_ipt(fig, FIGURES / '02_coortes_tratamento.png')
fig.show()

,ano_coorte_candidata,n_candidatos
0,2009,21
1,2010,27
2,2011,66
3,2012,13
4,2013,2


### Interpretação

Municípios de uma mesma coorte compartilham o mesmo ano candidato de tratamento. A maior concentração ocorre em 2011; as coortes de 2012 e 2013 são pequenas e exigirão cautela em qualquer avaliação posterior de heterogeneidade.

## 6. Linha do tempo

A linha do tempo abaixo situa as coortes dentro da janela observada. Para a coorte de 2009, apenas 2007 e 2008 estão disponíveis como anos pré-tratamento.

In [5]:
fig = go.Figure()
for _, linha in coortes.iterrows():
    rotulo = f"Coorte {linha['ano_coorte_candidata']}"
    fig.add_trace(go.Scatter(x=[2007, 2019], y=[rotulo, rotulo], mode='lines', line={'color': CORES_IPT['CINZA_GRADE'], 'width': 2}, hoverinfo='skip', showlegend=False))
    fig.add_trace(go.Scatter(x=[linha['ano_coorte_candidata']], y=[rotulo], mode='markers+text', marker={'color': CORES_IPT['AZUL_PRINCIPAL'], 'size': 14}, text=[f"{linha['n_candidatos']} candidatos"], textposition='top center', hovertemplate=f"{rotulo}<br>Início: %{{x}}<br>{linha['n_candidatos']} candidatos<extra></extra>", showlegend=False))
aplicar_tema_ipt(fig, titulo='Linha do tempo das coortes de tratamento')
fig.update_xaxes(title='Ano', tickmode='linear', dtick=1, range=[2006.6, 2019.4])
fig.update_yaxes(title='', showgrid=False, categoryorder='array', categoryarray=[f'Coorte {ano}' for ano in coortes['ano_coorte_candidata'][::-1]])
fig.add_annotation(x=2008, y='Coorte 2009', text='Somente 2007 e 2008<br>antes do tratamento', showarrow=True, arrowhead=2, ax=55, ay=-45, bgcolor=CORES_IPT['CINZA_FUNDO'], bordercolor=CORES_IPT['CINZA_GRADE'], font={'color': CORES_IPT['AZUL_ESCURO']})
salvar_figura_ipt(fig, FIGURES / '03_linha_tempo_coortes.png')
fig.show()

### Interpretação

A posição da coorte no início da janela de observação limita quantos anos pré-tratamento podem ser avaliados. Isso é uma propriedade do calendário do estudo, não uma evidência de efeito nem de ausência de efeito.

## 7. Suporte temporal dos tratados

O D12 exige janelas adjacentes completas: 2 pré + 3 pós requer `g-2` a `g+2`; 3 pré + 3 pós requer `g-3` a `g+2`. Todos os anos requeridos devem existir no painel e ter CEMPRE 708 numérico utilizável.

In [6]:
suporte_tratados = resumo_coortes[['ano_coorte_candidata', 'n_candidatos', 'n_elegivel_diag_2pre_3pos', 'n_elegivel_diag_3pre_3pos']].copy()
suporte_tratados['ano_coorte_candidata'] = suporte_tratados['ano_coorte_candidata'].astype(int)
display(estilizar_tabela_ipt(suporte_tratados))
fig = go.Figure()
fig.add_bar(name='2 pré + 3 pós', x=suporte_tratados['ano_coorte_candidata'].astype(str), y=suporte_tratados['n_elegivel_diag_2pre_3pos'], marker_color=CORES_IPT['AZUL_PRINCIPAL'], text=suporte_tratados['n_elegivel_diag_2pre_3pos'], textposition='outside', hovertemplate='Coorte %{x}<br>2 pré + 3 pós: %{y}<extra></extra>')
fig.add_bar(name='3 pré + 3 pós', x=suporte_tratados['ano_coorte_candidata'].astype(str), y=suporte_tratados['n_elegivel_diag_3pre_3pos'], marker_color=CORES_IPT['CIANO'], text=suporte_tratados['n_elegivel_diag_3pre_3pos'], textposition='outside', hovertemplate='Coorte %{x}<br>3 pré + 3 pós: %{y}<extra></extra>')
aplicar_tema_ipt(fig, titulo='Suporte temporal dos candidatos principais')
fig.update_layout(barmode='group')
fig.update_xaxes(title='Ano da coorte', type='category')
fig.update_yaxes(title='Número de municípios elegíveis', rangemode='tozero')
salvar_figura_ipt(fig, FIGURES / '04_suporte_temporal_tratados.png')
fig.show()

,ano_coorte_candidata,n_candidatos,n_elegivel_diag_2pre_3pos,n_elegivel_diag_3pre_3pos
0,2009,21,21,0
1,2010,27,27,27
2,2011,66,66,66
3,2012,13,13,13
4,2013,2,2,2


### Interpretação

A janela de 2 pré + 3 pós está disponível para todos os 129 candidatos. Já 3 pré + 3 pós não é possível para os 21 municípios da coorte de 2009, porque exigiria 2006, fora do painel 2007–2019. Esta limitação é de calendário, não de missing do outcome.

## 8. Disponibilidade do outcome

A tabela avalia a disponibilidade de CEMPRE 708 nas 1.677 observações dos 129 candidatos. O outcome não foi transformado e nenhum log foi aplicado.

In [7]:
candidatos = diagnostico.loc[diagnostico['candidato_amostra_principal'] == True]
disponibilidade_outcome = pd.DataFrame({
    'categoria': ['Total', 'Observado', 'Missing', 'Sigilo', 'Indisponível', 'Zero'],
    'n_observacoes': [
        candidatos['n_708_total'].sum(), candidatos['n_708_observado'].sum(), candidatos['n_708_missing'].sum(),
        candidatos['n_708_sigilo'].sum(), candidatos['n_708_indisponivel'].sum(), candidatos['n_708_zero'].sum(),
    ],
})
display(estilizar_tabela_ipt(disponibilidade_outcome))

,categoria,n_observacoes
0,Total,1677
1,Observado,1677
2,Missing,0
3,Sigilo,0
4,Indisponível,0
5,Zero,0


### Interpretação

A disponibilidade do outcome nos candidatos principais é uma verificação de qualidade de mensuração, não uma transformação analítica. Ela permite separar limitações do outcome de limitações puramente temporais do painel.

## 9. Controles por coorte

O diagnóstico aplica a mesma regra de janela adjacente aos 4.964 controles estruturalmente elegíveis. Os 4.970 municípios nunca expostos são um conjunto diferente e mais amplo.

In [8]:
controles_exibicao = controles_coortes[['ano_coorte_candidata', 'controles_2pre_3pos', 'controles_3pre_3pos']].copy()
controles_exibicao['ano_coorte_candidata'] = controles_exibicao['ano_coorte_candidata'].astype(int)
display(estilizar_tabela_ipt(controles_exibicao))
fig = go.Figure()
fig.add_bar(name='2 pré + 3 pós', x=controles_exibicao['ano_coorte_candidata'].astype(str), y=controles_exibicao['controles_2pre_3pos'], marker_color=CORES_IPT['AZUL_PRINCIPAL'], text=controles_exibicao['controles_2pre_3pos'], textposition='outside', hovertemplate='Coorte %{x}<br>2 pré + 3 pós: %{y:,.0f}<extra></extra>')
fig.add_bar(name='3 pré + 3 pós', x=controles_exibicao['ano_coorte_candidata'].astype(str), y=controles_exibicao['controles_3pre_3pos'], marker_color=CORES_IPT['CIANO'], text=controles_exibicao['controles_3pre_3pos'], textposition='outside', hovertemplate='Coorte %{x}<br>3 pré + 3 pós: %{y:,.0f}<extra></extra>')
aplicar_tema_ipt(fig, titulo='Controles estruturais com janela adjacente completa')
fig.update_layout(barmode='group')
fig.update_xaxes(title='Ano da coorte', type='category')
fig.update_yaxes(title='Número de controles', rangemode='tozero')
salvar_figura_ipt(fig, FIGURES / '05_controles_por_coorte.png')
fig.show()

,ano_coorte_candidata,controles_2pre_3pos,controles_3pre_3pos
0,2009,4964,0
1,2010,4963,4963
2,2011,4963,4963
3,2012,4963,4963
4,2013,4963,4963


### Interpretação

A única célula sigilosa do pool estrutural é o município 5003900 em 2012. Ela não afeta a coorte de 2009, cuja janela termina em 2011; afeta as coortes de 2010 a 2013, pois 2012 pertence às suas janelas adjacentes. Este diagnóstico não seleciona nem persiste uma amostra causal final.

## 10. O que já sabemos

- O painel nacional município-ano foi construído.
- O cadastro institucional foi reconstruído.
- O tratamento é escalonado.
- Há 129 candidatos principais e 4.964 controles estruturais.
- O outcome CEMPRE 708 está diagnosticado.
- O suporte temporal foi mensurado com janelas adjacentes explícitas.

## 11. O que ainda não sabemos

Ainda **não** sabemos se o desenho causal é defensável. Faltam decidir ou verificar: grupo final de comparação, antecipação, pré-tendências, comparabilidade, especificação do event study, estimador causal e robustez.

`AMOSTRA_CAUSAL_FINAL = NÃO DEFINIDA`

`DESENHO_CAUSAL_APROVADO = NÃO`

## 12. Próxima etapa

O próximo passo é o gate de identificação causal. Depois dele, se aprovado: diagnóstico de pré-tendências → definição da especificação → estimação → robustez → interpretação. Nenhuma dessas etapas é executada neste notebook.